# 面试问题：什么时候应该使用固定 Workflow，什么时候才需要 Agent？

可以直接复述的回答是：第一，先看任务路径是否稳定，而不是先看模型能力。第二，步骤确定、输入结构化且失败代价高的任务优先使用可审计 Workflow。第三，目标明确但路径依赖现场信息、工具选择较多时才引入 Agent。第四，高风险写操作无论走哪条路线都要经过确定性策略和人工审批。第五，评估时必须在同一批请求上比较完成率、错误路由、成本与延迟。第六，Agent 不是 Workflow 的替代品，生产系统通常是“确定性外壳包住受限 Agent”。下面用企业 IT 服务台的真实字段验证这些判断。

## 真实案例：企业 IT 服务台的任务路由

案例使用 6 条脱敏离线请求，字段结构参考常见工单系统：请求文本、路径是否已知、歧义数量、候选工具数和操作风险。目标是把请求路由到固定流程、受限 Agent 或带审批的流程。数据是教学实验中的结构化样本，不代表任何公司的线上分布；真实系统还需要历史完成率、租户权限和实时工具健康度。

In [1]:
from pprint import pprint  # 使用标准库格式化展示脱敏工单，不依赖 Agent 框架
requests = [  # 构造六条具有真实决策字段的 IT 服务台请求
    {"id": "IT-101", "text": "重置我的 VPN 密码", "known_flow": True, "ambiguity": 0, "tool_options": 1, "risk": "low"},  # 密码重置有稳定 SOP 且风险较低
    {"id": "IT-102", "text": "新员工入职，需要邮箱、代码库和项目空间", "known_flow": True, "ambiguity": 1, "tool_options": 3, "risk": "medium"},  # 入职流程固定但包含多个系统
    {"id": "IT-103", "text": "线上接口变慢，请定位原因并给出处理建议", "known_flow": False, "ambiguity": 3, "tool_options": 6, "risk": "medium"},  # 故障诊断需要根据观察动态选工具
    {"id": "IT-104", "text": "删除已离职员工账号并转移资料", "known_flow": True, "ambiguity": 1, "tool_options": 3, "risk": "high"},  # 删除和资料转移必须绑定审批
    {"id": "IT-105", "text": "查一下本月云资源账单异常项", "known_flow": False, "ambiguity": 2, "tool_options": 5, "risk": "low"},  # 账单分析路径依赖异常分布
    {"id": "IT-106", "text": "查询报修单 IT-088 的当前状态", "known_flow": True, "ambiguity": 0, "tool_options": 1, "risk": "low"},  # 状态查询是单一只读调用
]  # 结束脱敏请求集合
print("输入预览：id | 风险 | 歧义 | 工具数 | 用户请求")  # 给学习者展示路由器真正接收的字段
for request in requests:  # 逐条输出六个有业务语义的请求
    print(f"{request['id']} | {request['risk']:6} | {request['ambiguity']} | {request['tool_options']} | {request['text']}")  # 用紧凑表格呈现原始输入


输入预览：id | 风险 | 歧义 | 工具数 | 用户请求
IT-101 | low    | 0 | 1 | 重置我的 VPN 密码
IT-102 | medium | 1 | 3 | 新员工入职，需要邮箱、代码库和项目空间
IT-103 | medium | 3 | 6 | 线上接口变慢，请定位原因并给出处理建议
IT-104 | high   | 1 | 3 | 删除已离职员工账号并转移资料
IT-105 | low    | 2 | 5 | 查一下本月云资源账单异常项
IT-106 | low    | 0 | 1 | 查询报修单 IT-088 的当前状态


## Baseline / 基线：仅靠关键词选路线

最简单的做法是维护一组“固定流程关键词”，命中就走 Workflow，否则交给 Agent。它便宜且容易解释，但完全忽略歧义、工具数量和操作风险。

In [2]:
workflow_keywords = ("重置", "入职", "删除", "查询报修单")  # 用四个关键词模拟常见的硬编码路由表
def baseline_route(request):  # 实现只看请求文本的最小路由基线
    if any(keyword in request["text"] for keyword in workflow_keywords):  # 命中固定词时直接走流程
        return "workflow"  # 返回固定工作流而不检查风险
    return "agent"  # 未命中关键词时全部交给 Agent
baseline_rows = []  # 收集每条请求的基线路由结果
for request in requests:  # 在同一批六条请求上运行基线
    baseline_rows.append((request["id"], baseline_route(request)))  # 保存请求编号与路线
print("基线路由：id | route")  # 明确输出最简单方案的结果
for request_id, route in baseline_rows:  # 逐条展示关键词路由结果
    print(f"{request_id} | {route}")  # 让错误路由可被肉眼发现


基线路由：id | route
IT-101 | workflow
IT-102 | workflow
IT-103 | agent
IT-104 | workflow
IT-105 | agent
IT-106 | workflow


## 核心实现：受风险约束的路由决策

先由确定性策略处理高风险写操作，再根据路径稳定性和歧义决定 Workflow 或 Agent。这里同时输出决策分数与理由，避免只留下一个不可解释标签。

In [3]:
def governed_route(request):  # 实现风险优先、复杂度次之的受治理路由器
    complexity = request["ambiguity"] + max(0, request["tool_options"] - 2)  # 把歧义与候选工具数量合成可解释复杂度
    if request["risk"] == "high":  # 先截获删除账号等高风险操作
        return "workflow+approval", complexity, "高风险写操作必须审批"  # 用确定性流程承载审批与审计
    if request["known_flow"] and complexity <= 2:  # 已知路径且变化较少时不需要 Agent
        return "workflow", complexity, "步骤稳定且输入充分"  # 选择成本更低的固定流程
    return "bounded-agent", complexity, "路径依赖观察，允许受限选工具"  # 只把开放诊断交给受限 Agent
governed_rows = []  # 收集路线、分数和可审计原因
for request in requests:  # 对六条相同请求运行核心策略
    route, score, reason = governed_route(request)  # 计算当前请求的受治理路线
    governed_rows.append((request["id"], route, score, reason))  # 保存完整决策证据
print("受治理路由：id | route | complexity | reason")  # 输出策略最关键的中间过程
for row in governed_rows:  # 逐条展示路线与决策原因
    print(" | ".join(map(str, row)))  # 将四个字段格式化为可读审计表


受治理路由：id | route | complexity | reason
IT-101 | workflow | 0 | 步骤稳定且输入充分
IT-102 | workflow | 2 | 步骤稳定且输入充分
IT-103 | bounded-agent | 7 | 路径依赖观察，允许受限选工具
IT-104 | workflow+approval | 2 | 高风险写操作必须审批
IT-105 | bounded-agent | 5 | 路径依赖观察，允许受限选工具
IT-106 | workflow | 0 | 步骤稳定且输入充分


## 失败案例与修正：删除账号不能因“路径固定”而自动执行

关键词基线把 IT-104 当成普通 Workflow，意味着流程可能在没有审批票据时执行删除。修正方式不是把整个请求交给更强的 Agent，而是在路线选择之前设置不可绕过的风险门禁。

In [4]:
dangerous = next(request for request in requests if request["id"] == "IT-104")  # 取出删除离职账号的高风险反例
unsafe_route = baseline_route(dangerous)  # 运行忽略风险的关键词基线
safe_route, safe_score, safe_reason = governed_route(dangerous)  # 运行风险优先的修正策略
unsafe_action = "直接进入删除步骤" if unsafe_route == "workflow" else "交给 Agent"  # 翻译基线标签为真实失败语义
safe_action = "暂停并等待审批票据" if safe_route == "workflow+approval" else "继续执行"  # 翻译修正路线为执行行为
print("失败案例：", dangerous["text"])  # 展示触发风险的原始用户请求
print(f"修正前：{unsafe_route} -> {unsafe_action}")  # 显示基线可能造成的危险动作
print(f"修正后：{safe_route} -> {safe_action}，原因：{safe_reason}")  # 显示门禁后的安全动作和依据


失败案例： 删除已离职员工账号并转移资料
修正前：workflow -> 直接进入删除步骤
修正后：workflow+approval -> 暂停并等待审批票据，原因：高风险写操作必须审批


## 结果表：同一批请求上的路由准确率

教学实验给每条请求指定期望路线，用来比较关键词基线和受治理策略。这里的准确率只说明规则是否覆盖这六种已知场景，不代表线上泛化能力。

In [5]:
expected = {"IT-101": "workflow", "IT-102": "workflow", "IT-103": "bounded-agent", "IT-104": "workflow+approval", "IT-105": "bounded-agent", "IT-106": "workflow"}  # 定义六个业务场景的人工期望路线
baseline_correct = 0  # 初始化关键词基线命中数
governed_correct = 0  # 初始化受治理策略命中数
print("逐样本对照：id | expected | baseline | governed")  # 输出可直接检查的路线比较表
for request in requests:  # 在同一数据上比较两种方案
    baseline = baseline_route(request)  # 获取关键词基线路线
    governed = governed_route(request)[0]  # 获取受治理策略路线
    baseline_correct += int(baseline == expected[request["id"]])  # 累加基线正确样本数
    governed_correct += int(governed == expected[request["id"]])  # 累加核心策略正确样本数
    print(f"{request['id']} | {expected[request['id']]} | {baseline} | {governed}")  # 展示每条请求的差异来源
baseline_accuracy = baseline_correct / len(requests)  # 计算关键词基线准确率
governed_accuracy = governed_correct / len(requests)  # 计算受治理策略准确率
print(f"汇总：baseline={baseline_accuracy:.1%}，governed={governed_accuracy:.1%}")  # 输出同指标下的最终对照


逐样本对照：id | expected | baseline | governed
IT-101 | workflow | workflow | workflow
IT-102 | workflow | workflow | workflow
IT-103 | bounded-agent | agent | bounded-agent
IT-104 | workflow+approval | workflow | workflow+approval
IT-105 | bounded-agent | agent | bounded-agent
IT-106 | workflow | workflow | workflow
汇总：baseline=50.0%，governed=100.0%


## 结果解读

关键词基线在简单查询上有效，但把复杂诊断粗暴地交给 Agent，也漏掉了删除账号的审批语义。受治理策略的关键收益不是“更智能”，而是让高风险动作先受策略约束，让开放式推理只发生在允许的范围内。复杂度分数只是教学代理变量；线上应从历史轨迹校准阈值，并单独监控人工接管率和错误写操作率。

## 生产边界

真实服务台还需要租户 ACL、工具健康检查、审批系统签名、模型超时与降级、策略版本号以及人工标注的离线评测集。这里没有调用真实工单 API，也没有模拟并发和提示注入，因此不能把 100% 教学准确率外推到生产。上线时应让路由决策写入事件账本，并对每次工具调用再次鉴权。

## 最小回归测试

最后仅保护高风险门禁、两类稳定路线和本次教学对照的核心结论。

In [6]:
assert len(requests) >= 5  # 保证案例至少包含五条可读业务请求
assert governed_route(dangerous)[0] == "workflow+approval"  # 保证删除账号始终进入审批流程
assert governed_route(requests[0])[0] == "workflow"  # 保证稳定的密码重置不滥用 Agent
assert governed_route(requests[2])[0] == "bounded-agent"  # 保证开放式故障诊断进入受限 Agent
assert governed_accuracy > baseline_accuracy  # 保证修正策略在同一教学集上优于关键词基线
